In [10]:
import torch
import librosa
from pathlib import Path
from IPython.display import Audio
from pipeline import build_audiosep, separate_audio

In [14]:
checkpoints_dir = Path("checkpoint")
checkpoints_dir.mkdir(exist_ok=True)

models = (
    (
        "https://huggingface.co/spaces/badayvedat/AudioSep/resolve/main/checkpoint/audiosep_base_4M_steps.ckpt",
        checkpoints_dir / "audiosep_base_4M_steps.ckpt"
    ),
    (
        "https://huggingface.co/spaces/badayvedat/AudioSep/resolve/main/checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt",
        checkpoints_dir / "music_speech_audioset_epoch_15_esc_89.98.pt"
    )
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = build_audiosep(
      config_yaml='config/audiosep_base.yaml',
      checkpoint_path=str(models[0][1]),
      device=device)

audio_file = '/data/mtseng/atc_datasets/atc-dataset/data/wav/test_41.wav'
text = 'background noise'
output_file='noise.wav'

# AudioSep processes the audio at 32 kHz sampling rate
separate_audio(model, audio_file, text, output_file, device)

text = 'human speech'
output_file='speech.wav'
separate_audio(model, audio_file, text, output_file, device)

/data/mtseng/miniconda3/envs/atc_generation/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.dense.weight', 'lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some we

Loaded AudioSep model from [checkpoint/audiosep_base_4M_steps.ckpt]
Separating audio from [/data/mtseng/atc_datasets/atc-dataset/data/wav/test_41.wav] with textual query: [background noise]
Separated audio written to [noise.wav]
Separating audio from [/data/mtseng/atc_datasets/atc-dataset/data/wav/test_41.wav] with textual query: [human speech]
Separated audio written to [speech.wav]


In [11]:
noisy_file = 'noisy_audio.wav'

original_audio, _ = librosa.load(audio_file)
clean_file, _ = librosa.load(output_file)
noisy_audio = original_audio - clean_file

display(Audio(original_audio, rate=32000))